O objetivo do notebook é entender se as regras que o AVIA já utiliza para orientar a captura de fotos (heurísticas de visão computacional) funcionam bem, onde falham e se o problema pode ser corrigido ajustando seus limites ou se é necessário desenvolver uma forma melhor de analisar as imagens.

- **matriz de confusão, precisão e revocação** avaliam a *classificação* (um corte só);
- **ROC, AUC, curva de precisão-revocação e AP** avaliam o *score* (todos os cortes possíveis).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn 

base = pd.read_csv("https://raw.githubusercontent.com/joyceximenes/assistente_IA/main/analysis/data/guidance_scores.csv")

base.columns

In [ ]:
base.head()


| Colunas | Papel | De onde vêm |
|---|---|---|
| `split`, `caminho` | identificação: treino ou validação; qual foto | manifesto do dataset |
| `BLR`, `BRT`, `DRK`, `FRM` | **rótulo humano** (a referência): 1 se pelo menos 2 de 5 anotadores marcaram o problema | VizWiz |
| `ok`, `message` | decisão final do `guidance.ts` (ele para no primeiro problema que encontra) | app |
| `blurScore`, `edgeScore`, `brightnessScore`, `overexposedRatio` | **scores brutos** que o `guidance.ts` calculou antes de decidir | app |

Os scores brutos permitem avaliar cada problema isolado e, nas curvas ROC e P×R, testar *todos* os cortes, não só o de hoje.


## Linha de base — guidance.ts × rótulo humano

Mede o quanto a heurística atual do app concorda com os rótulos dos anotadores humanos do VizWiz. Avaliado só no conjunto de **validação**, pra manter esse dado intocado até a calibração futura.


In [ ]:
guidance_base = base[base.split == "val"].copy()
guidance_base.head(2)

### Selecionar o conjunto de validação

**Por que só a validação?** Os dados têm papéis diferentes no projeto:

| Conjunto | Papel |
|---|---|
| treino | Sprint 5: **calibrar** os cortes e treinar o classificador |
| validação | **medir** (este notebook); fica intocada até o "depois" |
| teste | o VizWiz de qualidade **não traz rótulos** de teste, então não existe |

Se o corte fosse escolhido olhando a validação, ela deixaria de ser uma medida independente: a informação da escolha vazaria para a nota. Daí a regra: **calibrar no treino, medir na validação**.


### BLR - BlurScore - desfoque
`blr_pred`: previsão do app pra cada foto, usando a regra de hoje do `guidance.ts` (desfocada se `blurScore < 120`, o valor de `BLUR_MIN`). Vira uma coluna nova em `guidance_base`, sem perder nenhuma linha — fica lado a lado com `BLR` (rótulo humano), pronta pra comparar na matriz de confusão.

In [ ]:
guidance_base["blr_pred"] = guidance_base.blurScore < 120
guidance_base.head(2)

### De score para previsão

`guidance_base.blurScore < 120` compara cada linha com 120 e devolve True/False; a atribuição guarda esse resultado na coluna nova `blr_pred`.

| Coluna | Tipo | O que significa |
|---|---|---|
| `blurScore` | número | **score**: quanto menor, mais desfocada |
| `blr_pred` | True/False | **previsão do app**: `True` = "desfocada" (`blurScore < 120`) |
| `BLR` | 0/1 | **rótulo humano**: 1 = pelo menos 2 de 5 anotadores acharam desfocada |



matriz de confusão para comparar BLR (rotulo verdadeiro, dado por humanos) com blr_pred (previsão do app) 

In [ ]:
# y_real, y_previsto
# BLR =  se é desfocado ou nao. validação humana
# blr_pred = o que o app achou em relaçao ao desfoque
comparative_blr = sklearn.metrics.confusion_matrix(guidance_base.BLR, guidance_base.blr_pred)
graph_blr = sklearn.metrics.ConfusionMatrixDisplay(comparative_blr).plot()

### Matriz de confusão

A matriz compara, foto a foto, o que os humanos disseram (linhas) com o que o app disse (colunas):

|  | App: não desfocada | App: desfocada |
|---|---|---|
| **Humano: nítida** | 3.687 — VN (acerto) | 784 — FP (alerta errado) |
| **Humano: desfocada** | 1.339 — FN (deixou passar) | 1.940 — VP (acerto) |

### O que cada erro significa no AVIA

| Erro | O que acontece com o usuário |
|---|---|
| FP (784) | o app pede para ajustar o foco de uma foto que já estava boa: atrasa a captura |
| FN (1.339) | o app não avisa sobre o desfoque: a foto ruim segue adiante |


In [ ]:
# precisão
presc_blr = sklearn.metrics.precision_score(guidance_base.BLR, guidance_base.blr_pred)

# revocação
rev_blr = sklearn.metrics.recall_score(guidance_base.BLR, guidance_base.blr_pred)

print(f"precisão: {presc_blr:.2%}, revocação: {rev_blr:.2%}")

In [ ]:
import numpy as np

# a few scores are below 1; clip them to 1 so they show on the log axis
blurry = guidance_base[guidance_base.BLR == 1].blurScore.clip(lower=1)
sharp = guidance_base[guidance_base.BLR == 0].blurScore.clip(lower=1)
bins = np.logspace(0, 4.1, 50)  # from 1 to ~12,600, evenly spaced on a log scale

plt.figure(figsize=(9, 4))
plt.hist(sharp, bins=bins, weights=np.ones(len(sharp)) / len(sharp), alpha=0.5, label="nítidas (BLR = 0)")
plt.hist(blurry, bins=bins, weights=np.ones(len(blurry)) / len(blurry), alpha=0.5, label="desfocadas (BLR = 1)")
plt.axvline(120, color="red", label="corte de hoje (120)")
plt.xscale("log")
plt.xlabel("blurScore (escala log)")
plt.ylabel("fração do grupo")
plt.legend()
plt.show()



### O que o gráfico mostra

- As desfocadas (laranja) ficam mais à esquerda, com `blurScore` baixo; as nítidas (azul), mais à direita.
- No meio do gráfico (mais ou menos entre 50 e 500), as duas cores se misturam: ali há fotos desfocadas e nítidas com o mesmo `blurScore`.
- O corte 120 cai justamente nessa mistura.

### A matriz de confusão dentro do gráfico

| Região | O que é | Quanto |
|---|---|---|
| laranja à esquerda da linha | desfocadas que o app alerta (VP) | 59 de cada 100 desfocadas |
| laranja à direita da linha | desfocadas que passam sem alerta (FN) | 41 de cada 100 desfocadas |
| azul à esquerda da linha | nítidas alertadas por engano (FP) | 18 de cada 100 nítidas |
| azul à direita da linha | nítidas que passam, corretamente (VN) | 82 de cada 100 nítidas |

**Conclusão:** como as duas cores se misturam perto do corte, qualquer corte vai errar algumas fotos. A pergunta seguinte é: mover a linha ajuda? A próxima célula testa quatro posições.

In [ ]:
rows = []
for cut in [60, 120, 200, 400]:
    pred = guidance_base.blurScore < cut  # the app's alarm under this cut
    rows.append(
        {
            "corte": cut,
            "revocação (TPR)": pred[guidance_base.BLR == 1].mean(),  # share of the blurry photos alarmed
            "falso positivo (FPR)": pred[guidance_base.BLR == 0].mean(),  # share of the sharp photos alarmed
            "precisão": guidance_base.BLR[pred].mean(),  # share of the alarms that are really blurry
        }
    )

cuts_blr = pd.DataFrame(rows).set_index("corte")
cuts_blr.round(3)

In [ ]:
fpr_blr, tpr_blr, thresholds_blr = sklearn.metrics.roc_curve(guidance_base.BLR, -guidance_base.blurScore)
# fpr_blr = taxa de falso positivo
# tpr_blr = taxa de verdadeiro positivo
# thresholds_blr = os cortes testados
roc_blr = sklearn.metrics.roc_auc_score(guidance_base.BLR, -guidance_base.blurScore)
print(f"Área sob a ROC: {roc_blr:.2}")

In [ ]:
plt.figure(figsize=(5, 5))
plt.plot(fpr_blr, tpr_blr, label=f"blurScore (AUC {roc_blr:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="acaso")
plt.scatter(cuts_blr["falso positivo (FPR)"], cuts_blr["revocação (TPR)"], color="red", zorder=5, label="os 4 cortes da tabela")
for cut, row in cuts_blr.iterrows():
    plt.annotate(f"  {cut}", (row["falso positivo (FPR)"], row["revocação (TPR)"]))
plt.xlabel("taxa de falso positivo")
plt.ylabel("revocação (taxa de verdadeiro positivo)")
plt.title("ROC — BLR")
plt.legend(loc="lower right")
plt.show()

In [ ]:
precision_blr, recall_blr, thresholds_pr_blr = sklearn.metrics.precision_recall_curve(guidance_base.BLR, -guidance_base.blurScore)
graph_curve_blr = sklearn.metrics.PrecisionRecallDisplay(precision=precision_blr, recall=recall_blr).plot()

In [ ]:
prevalence_blr = guidance_base.BLR.mean()

plt.figure(figsize=(5, 5))
plt.plot(recall_blr, precision_blr, label="blurScore")
plt.axhline(prevalence_blr, linestyle="--", color="gray", label=f"acaso (prevalência {prevalence_blr:.2f})")
plt.scatter(cuts_blr["revocação (TPR)"], cuts_blr["precisão"], color="red", zorder=5, label="os 4 cortes da tabela")
for cut, row in cuts_blr.iterrows():
    plt.annotate(f"  {cut}", (row["revocação (TPR)"], row["precisão"]))
plt.xlabel("revocação")
plt.ylabel("precisão")
plt.ylim(0, 1.05)
plt.title("Precisão-revocação — BLR")
plt.legend(loc="lower right")
plt.show()

In [ ]:
media_presc = sklearn.metrics.average_precision_score(guidance_base.BLR, -guidance_base.blurScore)
media_presc

In [ ]:
import sklearn.metrics


def plot_score_hist(y_true, score, cuts, xlabel, bins, xlog=False):
    """Histogram of the score for each label group (bars = share of that group), with today's cut(s) in red."""
    plt.figure(figsize=(9, 4))
    for value, name in [(0, "não tem o problema (0)"), (1, "tem o problema (1)")]:
        group = score[y_true == value]
        plt.hist(group, bins=bins, weights=np.ones(len(group)) / len(group), alpha=0.5, label=name)
    for i, cut in enumerate(cuts):
        plt.axvline(cut, color="red", label="corte de hoje" if i == 0 else None)
    if xlog:
        plt.xscale("log")
    plt.xlabel(xlabel)
    plt.ylabel("fração do grupo")
    plt.legend()
    plt.show()


def rule_point(y_true, y_pred):
    """Where the app's current rule sits: (false positive rate, recall, precision)."""
    tn, fp, fn, tp = sklearn.metrics.confusion_matrix(y_true, y_pred).ravel()
    return fp / (fp + tn), tp / (tp + fn), tp / (tp + fp)


def plot_curves(y_true, scores, point, title):
    """ROC (left) and precision-recall (right) for one or more scores, with the app's rule marked."""
    fpr_rule, recall_rule, precision_rule = point
    fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(11, 4.5))
    for label, score in scores.items():
        fpr, tpr, _ = sklearn.metrics.roc_curve(y_true, score)
        precision, recall, _ = sklearn.metrics.precision_recall_curve(y_true, score)
        auc = sklearn.metrics.roc_auc_score(y_true, score)
        ap = sklearn.metrics.average_precision_score(y_true, score)
        ax_roc.plot(fpr, tpr, label=f"{label} (AUC {auc:.3f})")
        ax_pr.plot(recall, precision, label=f"{label} (AP {ap:.3f})")
    ax_roc.plot([0, 1], [0, 1], "--", color="gray", label="acaso")
    ax_pr.axhline(y_true.mean(), linestyle="--", color="gray", label=f"acaso ({y_true.mean():.3f})")
    ax_roc.scatter([fpr_rule], [recall_rule], color="red", zorder=5, label="regra de hoje")
    ax_pr.scatter([recall_rule], [precision_rule], color="red", zorder=5, label="regra de hoje")
    ax_roc.set(xlabel="taxa de falso positivo", ylabel="revocação (taxa de verdadeiro positivo)", title=f"ROC — {title}")
    ax_pr.set(xlabel="revocação", ylabel="precisão", title=f"Precisão-revocação — {title}")
    ax_roc.legend(loc="lower right")
    ax_pr.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

### DRK — fotos escuras

**Pergunta:** quando o app diz "ambiente muito escuro", ele está certo? E quantas fotos escuras ele deixa passar?

- **Regra do app:** `brightnessScore < 30`. O `brightnessScore` é o brilho médio da foto, de 0 (preto) a 255 (branco).
- **Só 6 de cada 100 fotos são escuras.** Por isso, o histograma mostra a parte de cada grupo (se mostrasse a contagem, as escuras quase não apareceriam), e o acaso da precisão é 6%, e não 50%.

In [ ]:
plot_score_hist(guidance_base.DRK, guidance_base.brightnessScore, [30], "brightnessScore", np.linspace(0, 255, 52))

In [ ]:
guidance_base["drk_pred"] = guidance_base.brightnessScore < 30

confusion_drk = sklearn.metrics.confusion_matrix(guidance_base.DRK, guidance_base.drk_pred)
graph_drk = sklearn.metrics.ConfusionMatrixDisplay(confusion_drk).plot()

In [ ]:
precision_drk = sklearn.metrics.precision_score(guidance_base.DRK, guidance_base.drk_pred)
recall_drk = sklearn.metrics.recall_score(guidance_base.DRK, guidance_base.drk_pred)
fpr_rule_drk = rule_point(guidance_base.DRK, guidance_base.drk_pred)[0]

print(f"precisão: {precision_drk:.2%}, revocação: {recall_drk:.2%}, falso positivo: {fpr_rule_drk:.2%}")
print(f"proporção de DRK na base (acaso da P×R): {guidance_base.DRK.mean():.2%}")
print(f"fotos que o app marca como escuras: {guidance_base.drk_pred.mean():.2%} | que os humanos marcaram: {guidance_base.DRK.mean():.2%}")
print("brightnessScore médio por rótulo humano:")
print(guidance_base.groupby("DRK").brightnessScore.mean().round(1))

In [ ]:
plot_curves(
    guidance_base.DRK,
    {"-brightnessScore": -guidance_base.brightnessScore},
    rule_point(guidance_base.DRK, guidance_base.drk_pred),
    "DRK",
)

In [ ]:
plot_score_hist(guidance_base.BRT, guidance_base.brightnessScore, [240], "brightnessScore", np.linspace(0, 255, 52))
plot_score_hist(guidance_base.BRT, guidance_base.overexposedRatio, [0.15], "overexposedRatio", np.linspace(0, 1, 51))

In [ ]:
guidance_base["brt_pred"] = (guidance_base.brightnessScore > 240) | (guidance_base.overexposedRatio > 0.15)

confusion_brt = sklearn.metrics.confusion_matrix(guidance_base.BRT, guidance_base.brt_pred)
graph_brt = sklearn.metrics.ConfusionMatrixDisplay(confusion_brt).plot()

In [ ]:
precision_brt = sklearn.metrics.precision_score(guidance_base.BRT, guidance_base.brt_pred)
recall_brt = sklearn.metrics.recall_score(guidance_base.BRT, guidance_base.brt_pred)
fpr_rule_brt = rule_point(guidance_base.BRT, guidance_base.brt_pred)[0]

print(f"precisão: {precision_brt:.2%}, revocação: {recall_brt:.2%}, falso positivo: {fpr_rule_brt:.2%}")
print(f"proporção de BRT na base (acaso da P×R): {guidance_base.BRT.mean():.2%}")

In [ ]:
by_brightness = guidance_base.brightnessScore > 240
by_overexposed = guidance_base.overexposedRatio > 0.15

print(f"alarmes só por brightnessScore > 240:    {by_brightness.sum()}")
print(f"alarmes só por overexposedRatio > 0.15:  {by_overexposed.sum()}")
print(f"alarmes da regra inteira (OU):           {guidance_base.brt_pred.sum()}")
print(f"alarmes de brightness que o overexposed NÃO pegou: {(by_brightness & ~by_overexposed).sum()}")
print("médias por rótulo humano:")
print(guidance_base.groupby("BRT")[["brightnessScore", "overexposedRatio"]].mean().round(3))

In [ ]:
plot_curves(
    guidance_base.BRT,
    {
        "overexposedRatio": guidance_base.overexposedRatio,
        "brightnessScore": guidance_base.brightnessScore,
    },
    rule_point(guidance_base.BRT, guidance_base.brt_pred),
    "BRT",
)

In [ ]:
plot_score_hist(guidance_base.FRM, guidance_base.edgeScore, [18, 55], "edgeScore", np.linspace(0, 65, 66))

In [ ]:
guidance_base["frm_pred"] = (guidance_base.edgeScore > 55) | (guidance_base.edgeScore < 18)

confusion_frm = sklearn.metrics.confusion_matrix(guidance_base.FRM, guidance_base.frm_pred)
graph_frm = sklearn.metrics.ConfusionMatrixDisplay(confusion_frm).plot()

In [ ]:
precision_frm = sklearn.metrics.precision_score(guidance_base.FRM, guidance_base.frm_pred)
recall_frm = sklearn.metrics.recall_score(guidance_base.FRM, guidance_base.frm_pred)
fpr_rule_frm = rule_point(guidance_base.FRM, guidance_base.frm_pred)[0]

print(f"precisão: {precision_frm:.2%}, revocação: {recall_frm:.2%}, falso positivo: {fpr_rule_frm:.2%}")
print(f"proporção de FRM na base (acaso da P×R): {guidance_base.FRM.mean():.2%}")
print(f"fotos que o app marca como mal enquadradas: {guidance_base.frm_pred.mean():.2%}")

# Does the high side (edgeScore > 55) matter, or can we score only the low side?
n_total = len(guidance_base)
n_low = (guidance_base.edgeScore < 18).sum()
n_high = (guidance_base.edgeScore > 55).sum()
print(f"edgeScore < 18: {n_low} fotos ({n_low / n_total:.1%}) | edgeScore > 55: {n_high} fotos ({n_high / n_total:.2%})")
print("edgeScore médio por rótulo humano:")
print(guidance_base.groupby("FRM").edgeScore.mean().round(2))

In [ ]:
plot_curves(
    guidance_base.FRM,
    {"-edgeScore": -guidance_base.edgeScore},
    rule_point(guidance_base.FRM, guidance_base.frm_pred),
    "FRM",
)

# Check of the simplification: a two-sided score (distance from the middle of the 18-55 band)
two_sided_score = (guidance_base.edgeScore - (18 + 55) / 2).abs()
auc_low_side = sklearn.metrics.roc_auc_score(guidance_base.FRM, -guidance_base.edgeScore)
auc_two_sided = sklearn.metrics.roc_auc_score(guidance_base.FRM, two_sided_score)
print(f"AUC com -edgeScore: {auc_low_side:.3f} | AUC com score de duas pontas: {auc_two_sided:.3f}")

In [ ]:
summary_rows = []
for category, pred_column, score in [
    ("BLR", "blr_pred", -guidance_base.blurScore),
    ("DRK", "drk_pred", -guidance_base.brightnessScore),
    ("BRT", "brt_pred", guidance_base.overexposedRatio),
    ("FRM", "frm_pred", -guidance_base.edgeScore),
]:
    y_true = guidance_base[category]
    y_pred = guidance_base[pred_column]
    fpr_rule, recall_rule, precision_rule = rule_point(y_true, y_pred)
    ap = sklearn.metrics.average_precision_score(y_true, score)
    summary_rows.append(
        {
            "categoria": category,
            "prevalência": y_true.mean(),
            "precisão": precision_rule,
            "revocação": recall_rule,
            "falso positivo": fpr_rule,
            "AUC": sklearn.metrics.roc_auc_score(y_true, score),
            "AP": ap,
            "AP ÷ prevalência": ap / y_true.mean(),
        }
    )

summary = pd.DataFrame(summary_rows).set_index("categoria").round(3)
summary

bootstrap - intervalo de confiança

In [ ]:
def bootstrap_metrics(y_true, y_pred, score, n_boot, rng):
    """Recompute precision, recall and AUC on n_boot resamples of the photos (drawn with replacement)."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    score = np.asarray(score)
    n_photos = len(y_true)
    rows = []
    for _ in range(n_boot):
        # same row index for label, prediction and score: the whole photo is drawn
        drawn = rng.integers(0, n_photos, size=n_photos)
        rows.append(
            {
                "precisão": sklearn.metrics.precision_score(y_true[drawn], y_pred[drawn]),
                "revocação": sklearn.metrics.recall_score(y_true[drawn], y_pred[drawn]),
                "AUC": sklearn.metrics.roc_auc_score(y_true[drawn], score[drawn]),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
rng = np.random.default_rng(42)
n_boot = 1000

ci_rows = []
for category, pred_column, score in [
    ("BLR", "blr_pred", -guidance_base.blurScore),
    ("DRK", "drk_pred", -guidance_base.brightnessScore),
    ("BRT", "brt_pred", guidance_base.overexposedRatio),
    ("FRM", "frm_pred", -guidance_base.edgeScore),
]:
    y_true = guidance_base[category]
    y_pred = guidance_base[pred_column]
    measured = {
        "precisão": sklearn.metrics.precision_score(y_true, y_pred),
        "revocação": sklearn.metrics.recall_score(y_true, y_pred),
        "AUC": sklearn.metrics.roc_auc_score(y_true, score),
    }
    resamples = bootstrap_metrics(y_true, y_pred, score, n_boot, rng)
    for metric, value in measured.items():
        low, high = np.percentile(resamples[metric], [2.5, 97.5])
        ci_rows.append(
            {"categoria": category, "métrica": metric, "medido": value, "IC 2,5%": low, "IC 97,5%": high}
        )

ci = pd.DataFrame(ci_rows).set_index(["categoria", "métrica"]).round(3)
ci

In [ ]:
# the measured values must match the summary table above (same photos, same scores): all True
metrics = ["precisão", "revocação", "AUC"]
ci["medido"].unstack()[metrics].eq(summary[metrics])